In [1]:
import os
import numpy as np
import pandas as pd
import healpy as hp
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord
from astroplan import Observer
import astropy.units as u

from rubin_nights import connections
from rubin_nights import lfa_data
from rubin_nights import plot_utils as rn_plots

from IPython.display import display, HTML
import matplotlib.pyplot as plt

import pickle
from lsst.resources import ResourcePath
from rubin_scheduler.scheduler.schedulers import CoreScheduler
from rubin_scheduler.scheduler.features import Conditions
from rubin_scheduler.scheduler.model_observatory import ModelObservatory
from rubin_scheduler.utils import ddf_locations

import rubin_scheduler.scheduler.basis_functions as basis_functions
import rubin_scheduler.scheduler.detailers as detailers
from rubin_scheduler.skybrightness_pre import dark_m5
from rubin_scheduler.site_models import SeeingModel

import schedview.compute as schedview_compute


In [2]:
endpoints_dev = connections.get_clients(tokenfile=os.path.join(os.path.expanduser("~"), ".lsst/usdf_rsp"), site='usdf-dev')
endpoints_dev

endpoints = endpoints_dev

In [3]:
#day_obs = Time(Time.now().mjd - 0.5, format='mjd', scale='tai').iso[0:10]
day_obs = "2026-06-29"

queue = 1


day_obs_time = Time(f"{day_obs}T12:00:00", format='isot', scale='tai')
tnow = Time.now()
observer = Observer.at_site('Rubin')
sunset = Time(observer.sun_set_time(day_obs_time, which='next', horizon=-0*u.deg), format='jd')
sunrise = Time(observer.sun_rise_time(day_obs_time, which='next', horizon=-0*u.deg), format='jd')
print(day_obs, 'sunset', sunset.iso,  'sunrise', sunrise.iso, 'now', Time.now().iso)

topic = "lsst.sal.Scheduler.logevent_target"
#topic = "lsst.sal.Scheduler.logevent_largeFileObjectAvailable"
targets = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
for c in ['ra', 'decl', 'skyAngle']:
    targets[c] = targets[c].astype(float)
if len(targets) == 0:
    display(endpoints['efd'].select_top_n(topic, '*', 2, index=queue))
print(len(targets))

topic = "lsst.sal.Scheduler.logevent_observation"
observations = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
for c in ['ra', 'decl', 'rotSkyPos']:
    observations[c] = observations[c].astype(float)
print(len(observations))

topic = "lsst.sal.Scheduler.logevent_largeFileObjectAvailable"
snapshots = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
print(len(snapshots))

if len(targets) > 0 and len(observations) > 0:
    to = pd.merge_asof(
                targets.sort_values("targetId").reset_index("time"),
                observations.sort_values("targetId").reset_index("time"),
                on="targetId",
                left_by=["ra", "decl", "skyAngle"],
                right_by=["ra", "decl", "rotSkyPos"],
                suffixes=("", "_o"),
                allow_exact_matches=True,
                direction="forward",
            )
    to.sort_values(by="time", inplace=True)
    to = to.astype({"targetId": int, "blockId": int, "skyAngle": float})
    print(len(to))
elif len(targets) > 0:
    to = pd.DataFrame(targets.sort_values("targetId").reset_index("time"))
    to['time_o'] = np.nan
    print("targets only")
else:
    to = None

2026-06-29 sunset 2026-06-29 21:48:43.224 sunrise 2026-06-30 11:44:28.099 now 2026-06-30 19:54:48.608
886
875
891
886


In [4]:
visits = endpoints['consdb'].get_visits("lsstcam", sunset, sunrise)

In [5]:
programs = ["BLOCK-365", "BLOCK-407", "BLOCK-408", "BLOCK-416"]
visits.query("science_program in @programs")[['target_name', 'visit_id', 'obs_start',  'observation_reason', 'science_program', 'band',  'zero_point_1s_pred', 'fwhm_eff', 'clouds']]


,target_name,visit_id,obs_start,observation_reason,science_program,band,zero_point_1s_pred,fwhm_eff,clouds
10,lowdust,2026062900045,2026-06-29T22:51:19.330000,twilight_near_sun,BLOCK-407,r,28.376665,2.027672,0.060814
11,"nes, lowdust",2026062900046,2026-06-29T22:51:47.622000,twilight_near_sun,BLOCK-407,r,28.360304,2.228412,0.082849
12,nes,2026062900047,2026-06-29T22:52:16.287000,twilight_near_sun,BLOCK-407,r,28.342320,2.060431,0.048547
13,lowdust,2026062900048,2026-06-29T22:52:52.253000,twilight_near_sun,BLOCK-407,r,28.350811,1.692259,0.008915
14,lowdust,2026062900049,2026-06-29T22:53:20.510000,twilight_near_sun,BLOCK-407,r,28.340926,1.790782,0.021362
...,...,...,...,...,...,...,...,...,...
880,nes,2026062900915,2026-06-30T10:42:10.439000,twilight_near_sun,BLOCK-407,r,28.361989,1.360721,-0.022051
881,nes,2026062900916,2026-06-30T10:42:39.872000,twilight_near_sun,BLOCK-407,r,28.358934,1.339038,-0.021399
882,nes,2026062900917,2026-06-30T10:43:07.478000,twilight_near_sun,BLOCK-407,r,28.343403,1.360780,-0.013822
883,nes,2026062900918,2026-06-30T10:43:38.636000,twilight_near_sun,BLOCK-407,r,28.354687,1.421419,0.016755


In [6]:
to.groupby("snapshotUri").first()[['time', 'ra', 'decl', 'skyAngle', 'filter', 'note', 'targetName', 'airmass', 'targetId', 'blockId']]


,time,ra,decl,skyAngle,filter,note,targetName,airmass,targetId,blockId
snapshotUri,,,,,,,,,,
,2026-06-29 22:15:16.864501+00:00,0.000000,0.000000,0.000000,,Target,Target,0.000000,0,117036
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-29T22:48:02.583.p,2026-06-29 22:47:32.223514+00:00,143.828146,11.892146,5.567092,r,"twilight_near_sun, 0",lowdust,2.006515,27432,117040
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-29T22:48:04.859.p,2026-06-29 22:47:33.465217+00:00,144.308708,14.995540,7.026352,r,"twilight_near_sun, 0","nes, lowdust",2.121625,27433,117041
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-29T22:48:06.926.p,2026-06-29 22:47:34.566638+00:00,144.907013,17.987898,8.418486,r,"twilight_near_sun, 0",nes,2.256742,27434,117042
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-29T22:48:23.793.p,2026-06-29 22:51:00.382168+00:00,141.484526,13.434876,4.781504,r,"twilight_near_sun, 0",lowdust,2.277628,27435,117043
...,...,...,...,...,...,...,...,...,...,...
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:41:32.938.p,2026-06-30 10:41:51.638449+00:00,51.508377,17.424714,94.803790,r,"twilight_near_sun, 1",nes,2.229324,23588,117920
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:41:34.978.p,2026-06-30 10:42:20.917902+00:00,52.289834,14.420172,96.234180,r,"twilight_near_sun, 1",nes,2.097080,23589,117921
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:43:00.211.p,2026-06-30 10:42:53.871477+00:00,55.931286,10.100800,99.678931,r,"twilight_near_sun, 1",nes,2.099900,23590,117922


In [12]:
#uri = to.iloc[1].snapshotUri

uri = "https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:43:04.371.p"

sched, conditions = lfa_data.get_scheduler_snapshot(uri, at_usdf=False)

ConnectTimeout: HTTPSConnectionPool(host='s3.cp.lsst.org', port=443): Max retries exceeded with url: /rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:43:04.371.p (Caused by ConnectTimeoutError(<HTTPSConnection(host='s3.cp.lsst.org', port=443) at 0x32aa07ed0>, 'Connection to s3.cp.lsst.org timed out. (connect timeout=60.0)'))

'https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-29T22:58:40.394.p'